# Compiled profile-coverage kinase screening

CPU-only reproducible experiment for **When Sequential Is Not Adaptive: Compiled Profile-Coverage Kinase Counter-Screening**.

The notebook tests whether correlation-aware coverage finds recorded kinase activities earlier than a matched similarity-weighted marginal ranker. It also checks lossless first-hit policy compilation, strong chemical-series exclusions, exact small-instance optima, support abstention, and latency-aware batches.

This is retrospective assay prioritization. It does not establish cellular engagement, toxicity, clinical safety, efficacy, or therapeutic suitability. PKIS2 lacks complete intended-target annotations and is therefore labeled **profile-activity discovery**, not off-target discovery.

**Runtime:** select a standard Colab CPU runtime. A GPU is unnecessary and is not used. **API cost:** USD 0.

In [ ]:
# Frozen execution controls. Leave SMOKE_TEST=False for paper results.
RUN_EXTERNAL_PKIS2 = True
SMOKE_TEST = False
BOOTSTRAP_REPLICATES = 10_000
SIGN_FLIP_PERMUTATIONS = 100_000
EXPECTED_BUNDLE = 'compiled_coverage_colab_bundle.zip'
PKIS2_URL = 'https://doi.org/10.1371/journal.pone.0181585.s004'
PKIS2_SHA256 = '48ead22a1f860cd0d5096fa87d5acd329f722fe8d65e693bb0be682a333e2a2c'
print('Configuration:', {
    'external_pkis2': RUN_EXTERNAL_PKIS2,
    'smoke_test': SMOKE_TEST,
    'bootstrap': BOOTSTRAP_REPLICATES,
    'permutations': SIGN_FLIP_PERMUTATIONS,
})

## 1. Upload and verify the compact project bundle

Upload only `compiled_coverage_colab_bundle.zip`. The 22 GB ChEMBL database is not needed. The compact derived ChEMBL records are redistributed under CC BY-SA 3.0.

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, time, urllib.request, zipfile

uploaded = files.upload()
assert EXPECTED_BUNDLE in uploaded, f'Upload {EXPECTED_BUNDLE}'
archive = Path('/content') / EXPECTED_BUNDLE
archive.write_bytes(uploaded[EXPECTED_BUNDLE])
with zipfile.ZipFile(archive) as zf:
    bad_member = zf.testzip()
    assert bad_member is None, f'Corrupt archive member: {bad_member}'
    zf.extractall('/content')
project = Path('/content/counterscreen_active_search')
assert project.is_dir(), 'Expected counterscreen_active_search/ in bundle'
print('Extracted:', project)

In [ ]:
# The method freeze is separate from the completed earlier study freezes.
freeze = subprocess.run(
    ['sha256sum', '-c', 'COMPILED_COVERAGE_FREEZE.sha256'],
    cwd=project, text=True, capture_output=True
)
print(freeze.stdout)
assert freeze.returncode == 0, freeze.stderr
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     str(project / 'requirements_method_colab.txt')],
    check=True
)
tests = subprocess.run(
    [sys.executable, 'src/test_compiled_coverage.py'],
    cwd=project, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(tests.stdout)
assert tests.returncode == 0, 'Deterministic method unit tests failed'
print('Frozen method files verified; CPU dependencies installed; unit tests passed.')

## 2. Validate the unchanged Klaeger/ChEMBL benchmark

In [ ]:
validation = subprocess.run(
    [sys.executable, 'src/validate_dataset.py', '--data-dir', 'data/derived',
     '--report', 'analysis/dataset_validation_compiled_coverage_colab.json'],
    cwd=project, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(validation.stdout)
assert validation.returncode == 0, 'Dataset validation failed'
assert '"status": "PASS"' in validation.stdout

## 3. Acquire and verify PKIS2 separately

The source is Drewry et al., PLOS ONE 2017, S4 Table (CC BY 4.0). Its 1 µM percent-inhibition measurements are never pooled with Klaeger `Kd` measurements. The exact source hash is checked before parsing.

In [ ]:
pkis2_path = Path('/content/pkis2_s4.xlsx')
if RUN_EXTERNAL_PKIS2:
    urllib.request.urlretrieve(PKIS2_URL, pkis2_path)
    observed = hashlib.sha256(pkis2_path.read_bytes()).hexdigest()
    print('PKIS2 SHA-256:', observed)
    assert observed == PKIS2_SHA256, 'Unexpected PKIS2 source bytes'
else:
    print('External PKIS2 evaluation disabled; this is not a submission-complete run.')

## 4. Run the frozen experiment

The complete run performs 10,000 paired bootstraps and 100,000 sign flips. Compiler equality is checked case-by-case with exact assertions, not p-values.

In [ ]:
output_dir = project / ('compiled_coverage_smoke_output' if SMOKE_TEST else 'compiled_coverage_output')
if output_dir.exists():
    shutil.rmtree(output_dir)
command = [
    sys.executable, 'src/run_compiled_coverage.py',
    '--data-dir', 'data/derived',
    '--output-dir', output_dir.name,
    '--max-budget', '20',
    '--bootstrap', str(BOOTSTRAP_REPLICATES),
    '--permutations', str(SIGN_FLIP_PERMUTATIONS),
]
if RUN_EXTERNAL_PKIS2:
    command.extend(['--pkis2-xlsx', str(pkis2_path)])
if SMOKE_TEST:
    command.append('--smoke-test')
started = time.time()
run = subprocess.run(
    command, cwd=project, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
output_dir.mkdir(exist_ok=True)
(output_dir / 'console.log').write_text(run.stdout, encoding='utf-8')
print(run.stdout[-20_000:])
assert run.returncode == 0, 'Experiment failed; inspect console.log above'
print(f'Wall time: {time.time() - started:.1f} seconds')

## 5. Inspect the decision results

A positive number favors correlation-aware weighted coverage. Confidence intervals crossing zero are inconclusive. Do not interpret a no-hit panel as biological safety.

In [ ]:
import pandas as pd
from IPython.display import display, Image

summary = json.loads((output_dir / 'summary.json').read_text(encoding='utf-8'))
assert summary['compiler']['all_checks_passed'] is True
assert summary['compiler']['toy_boundary']['multi_hit_counterexample_exists'] is True
print('Run status:', summary['status'])
print('Exact compiler case/policy/condition checks:',
      summary['compiler']['case_policy_condition_checks'])
primary = pd.DataFrame(summary['statistics']['primary_tests']).T
display(primary[['n', 'estimate', 'ci95', 'permutation_p_two_sided']])
display(pd.read_csv(output_dir / 'method_summary.csv'))
display(pd.read_csv(output_dir / 'support_abstention.csv'))
display(pd.read_csv(output_dir / 'exact_restricted_optima.csv').describe())
for figure in sorted((output_dir / 'figures').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure)))

## 6. Package every result and log

In [ ]:
result_archive = Path('/content/compiled_coverage_results.zip')
with zipfile.ZipFile(result_archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(output_dir.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(project))
    validation_report = project / 'analysis/dataset_validation_compiled_coverage_colab.json'
    if validation_report.exists():
        zf.write(validation_report, validation_report.relative_to(project))
    for name in [
        'COMPILED_COVERAGE_PROTOCOL.md',
        'COMPILED_COVERAGE_FREEZE.sha256',
        'requirements_method_colab.txt',
    ]:
        path = project / name
        zf.write(path, path.relative_to(project))
print('Result archive:', result_archive, result_archive.stat().st_size, 'bytes')
files.download(str(result_archive))